In [0]:
from pyspark.sql.functions import col, round, unix_timestamp

bronze_table_name = "nyc_taxi_etl.default.bronze_yellow_tripdata"
silver_table_name = "nyc_taxi_etl.default.silver_yellow_tripdata"

silver_table_path = "abfss://bronzedata@storagenyctaxietl.dfs.core.windows.net/silver/yellow_tripdata"

try:
    print("1. Reading from Bronze table...")
    bronze_df = spark.read.table(bronze_table_name)
    initial_count = bronze_df.count()
    
    print("2. Applying Silver Layer cleaning rules...")
    silver_df = bronze_df.filter(
        (col("trip_distance") > 0) & 
        (col("passenger_count") > 0) & 
        (col("total_amount") >= 0) &
        (col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    )
    
    silver_df = silver_df.dropDuplicates()
    
    print("3. Adding trip_duration_minutes column...")
    silver_df = silver_df.withColumn(
        "trip_duration_minutes",
        round((unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60, 2)
    )
    
    final_count = silver_df.count()
    print(f"   -> Dropped {initial_count - final_count} dirty rows.")
    
    print(f"4. Saving to Silver External Delta Table...")
    silver_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("path", silver_table_path) \
        .saveAsTable(silver_table_name)
        
    print(f"Success! {final_count} clean rows saved to Silver layer.")

except Exception as e:
    print(f"Silver layer processing failed: {e}")

1. Reading from Bronze table...
2. Applying Silver Layer cleaning rules...
3. Adding trip_duration_minutes column...
   -> Dropped 1188329 dirty rows.
4. Saving to Silver External Delta Table...
Success! 3240370 clean rows saved to Silver layer.
